In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
df = pd.read_csv("merged__panel.csv", parse_dates=["week_start", "week_end"])
df = df.sort_values(["district", "year", "week"]).reset_index(drop=True)


def add_lags(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    for lag in [1, 2, 3, 4]:
        g[f"cases_lag{lag}"] = g["cases"].shift(lag)
        g[f"precip_lag{lag}"] = g["precip"].shift(lag)
        g[f"t2m_lag{lag}"] = g["t2m"].shift(lag)
        g[f"rh2m_lag{lag}"] = g["rh2m"].shift(lag)
    g["cases_roll4"] = g["cases"].shift(1).rolling(4).mean()
    g["cases_roll8"] = g["cases"].shift(1).rolling(8).mean()
    g["precip_roll4"] = g["precip"].shift(1).rolling(4).mean()
    # forecast targets: cases 2 and 4 weeks AHEAD of the current row
    g["y_t2"] = g["cases"].shift(-2)
    g["y_t4"] = g["cases"].shift(-4)
    return g


feat = df.groupby("district", group_keys=False).apply(add_lags)
feat["week_sin"] = np.sin(2 * np.pi * feat["week"] / 52)
feat["week_cos"] = np.cos(2 * np.pi * feat["week"] / 52)
feat = feat.dropna().reset_index(drop=True)

FEATURE_COLS = [
    c for c in feat.columns
    if c.startswith(("cases_lag", "precip_lag", "t2m_lag", "rh2m_lag"))
    or c in ["cases_roll4", "cases_roll8", "precip_roll4", "week_sin", "week_cos"]
]

/tmp/ipykernel_1156/2749251883.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  feat = df.groupby("district", group_keys=False).apply(add_lags)


In [ ]:
train = feat[feat.year <= 2022]
val   = feat[feat.year == 2023]
test  = feat[(feat.year >= 2024) & (feat.year <= 2025)]  # 2026 excluded (partial year)

def rmse(y, p):
    return np.sqrt(mean_squared_error(y, p))

def score(y_true, y_pred, name):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "Pearson_r": pearsonr(y_true, y_pred)[0],
    }

all_results = []

In [ ]:
for horizon, ycol in [("t+2", "y_t2"), ("t+4", "y_t4")]:
    pred = test["cases_lag1"]
    r = score(test[ycol], pred, "Naive persistence")
    r["horizon"] = horizon
    all_results.append(r)


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

xgb_feature_cols = FEATURE_COLS + ["district_code"]
feat["district_code"] = feat["district"].astype("category").cat.codes

train_xgb = feat[feat.year <= 2022]
val_xgb   = feat[feat.year == 2023]
test_xgb  = feat[(feat.year >= 2024) & (feat.year <= 2025)]

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
}

for horizon, ycol in [("t+2", "y_t2"), ("t+4", "y_t4")]:
    base = XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1)
    # TimeSeriesSplit keeps CV folds chronological — no shuffling of time-ordered data
    search = RandomizedSearchCV(
        base, param_grid, n_iter=20, cv=TimeSeriesSplit(n_splits=4),
        scoring="neg_mean_absolute_error", random_state=RANDOM_STATE, n_jobs=-1,
    )
    search.fit(train_xgb[xgb_feature_cols], train_xgb[ycol])
    best = search.best_estimator_
    print(f"Best XGBoost params ({horizon}): {search.best_params_}")

    pred = best.predict(test_xgb[xgb_feature_cols])
    r = score(test_xgb[ycol], pred, "XGBoost (global, tuned)")
    r["horizon"] = horizon
    all_results.append(r)


Best XGBoost params (t+2): {'subsample': 0.7, 'n_estimators': 600, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
Best XGBoost params (t+4): {'subsample': 0.7, 'n_estimators': 600, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.7}


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

LOOKBACK = 8  # weeks of history fed into the LSTM

class DistrictSeqDataset(Dataset):
    def __init__(self, panel_df, horizon_col, lookback=LOOKBACK):
        self.samples = []
        seq_features = ["cases", "precip", "t2m", "rh2m"]
        for dist, d in panel_df.groupby("district"):
            d = d.sort_values(["year", "week"]).reset_index(drop=True)
            dist_code = d["district_code"].iloc[0]
            arr = d[seq_features].values.astype(np.float32)
            targets = d[horizon_col].values.astype(np.float32)
            for i in range(lookback, len(d)):
                if np.isnan(targets[i]):
                    continue
                self.samples.append((arr[i - lookback:i], dist_code, targets[i], d["year"].iloc[i]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq, dist_code, y, year = self.samples[idx]
        return torch.tensor(seq), torch.tensor(dist_code, dtype=torch.long), torch.tensor(y), year


class LSTMForecaster(nn.Module):
    def __init__(self, n_features=4, n_districts=25, emb_dim=8, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(n_districts, emb_dim)
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden + emb_dim, 32), nn.ReLU(), nn.Linear(32, 1)
        )

    def forward(self, seq, dist_code):
        _, (h, _) = self.lstm(seq)
        h = h.squeeze(0)
        e = self.emb(dist_code)
        return self.head(torch.cat([h, e], dim=1)).squeeze(-1)


def train_lstm(ycol, epochs=15, lr=1e-3, hidden=64, batch_size=128):
    ds = DistrictSeqDataset(feat, ycol)
    train_idx = [i for i, s in enumerate(ds.samples) if s[3] <= 2022]
    test_idx  = [i for i, s in enumerate(ds.samples) if 2024 <= s[3] <= 2025]

    train_loader = DataLoader(torch.utils.data.Subset(ds, train_idx), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(torch.utils.data.Subset(ds, test_idx), batch_size=batch_size, shuffle=False)

    model = LSTMForecaster(hidden=hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for seq, dist_code, y, _ in train_loader:
            opt.zero_grad()
            pred = model(seq, dist_code)
            loss = loss_fn(pred, y)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(y)
        print(f"  epoch {epoch+1}/{epochs}  train MSE={total_loss/len(train_idx):.2f}")

    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for seq, dist_code, y, _ in test_loader:
            preds.append(model(seq, dist_code).numpy())
            truths.append(y.numpy())
    return np.concatenate(preds), np.concatenate(truths)

In [ ]:
lstm_grid = [
    {"hidden": 32, "lr": 1e-3},
    {"hidden": 64, "lr": 1e-3},
    {"hidden": 64, "lr": 5e-4},
]

for horizon, ycol in [("t+2", "y_t2"), ("t+4", "y_t4")]:
    best_mae, best_result = np.inf, None
    for params in lstm_grid:
        print(f"LSTM {horizon} trying {params}")
        preds, truths = train_lstm(ycol, epochs=10, **params)
        mae = mean_absolute_error(truths, preds)
        if mae < best_mae:
            best_mae = mae
            best_result = score(truths, preds, "LSTM (global, tuned)")
    best_result["horizon"] = horizon
    all_results.append(best_result)

LSTM t+2 trying {'hidden': 32, 'lr': 0.001}
  epoch 1/10  train MSE=7603.95
  epoch 2/10  train MSE=5791.01
  epoch 3/10  train MSE=4571.90
  epoch 4/10  train MSE=3884.36
  epoch 5/10  train MSE=3472.82
  epoch 6/10  train MSE=3217.97
  epoch 7/10  train MSE=3031.97
  epoch 8/10  train MSE=2885.90
  epoch 9/10  train MSE=2763.79
  epoch 10/10  train MSE=2657.59
LSTM t+2 trying {'hidden': 64, 'lr': 0.001}
  epoch 1/10  train MSE=7079.78
  epoch 2/10  train MSE=4875.26
  epoch 3/10  train MSE=3805.37
  epoch 4/10  train MSE=3274.10
  epoch 5/10  train MSE=2983.25
  epoch 6/10  train MSE=2798.23
  epoch 7/10  train MSE=2659.72
  epoch 8/10  train MSE=2606.92
  epoch 9/10  train MSE=2478.14
  epoch 10/10  train MSE=2472.58
LSTM t+2 trying {'hidden': 64, 'lr': 0.0005}
  epoch 1/10  train MSE=7840.99
  epoch 2/10  train MSE=6587.71
  epoch 3/10  train MSE=5556.50
  epoch 4/10  train MSE=4830.69
  epoch 5/10  train MSE=4300.04
  epoch 6/10  train MSE=3917.91
  epoch 7/10  train MSE=3614.38
 

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

def sarimax_forecast_district(district_df, horizon_steps,
                               exog_cols=("precip", "t2m", "rh2m", "week_sin", "week_cos")):
    """Fit SARIMAX once on data through 2023, then roll forward through the
    test period using `.append(..., refit=False)` to update the filter state
    with real observed values (fast — no re-optimization), forecasting
    `horizon_steps` ahead at each point.

    Uses purely POSITIONAL (.iloc) logic internally to avoid any index-label
    mismatches, then maps predictions back to district_df's original row
    labels at the end so the caller can safely do d.loc[p.index, ycol].

    NOTE ON EXOG: this uses the *actual* observed future weather as exog,
    which is a common simplification for a baseline (real deployment would
    need forecasted weather, e.g. from a met-office forecast, for horizon_steps
    ahead). Flag this assumption explicitly in your proposal's limitations.

    Seasonality: instead of an explicit 52-week seasonal ARIMA term (very
    expensive to fit), seasonality is captured via the week_sin/week_cos
    Fourier terms as exogenous regressors — standard practice for weekly
    epi series with long seasonal periods, and far cheaper computationally.
    """
    d_sorted = district_df.sort_values(["year", "week"])
    orig_labels = d_sorted.index.to_numpy()          # original global row labels, in order
    d = d_sorted.reset_index(drop=True)               # d.index is now clean 0..N-1, purely positional

    train_mask = (d["year"] <= 2023).to_numpy()
    n_train = int(train_mask.sum())
    y_train = d.loc[train_mask, "cases"]
    exog_train = d.loc[train_mask, list(exog_cols)]

    model = SARIMAX(y_train.to_numpy(), exog=exog_train.to_numpy(), order=(2, 1, 1),
                     enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)

    test_mask = ((d["year"] >= 2024) & (d["year"] <= 2025)).to_numpy()
    test_positions = np.where(test_mask)[0]           # positional integer locations, 0..N-1

    preds = []
    current_fit = fit
    last_appended = n_train - 1

    for pos in test_positions:
        # bring the filter state up to date with any real observations since the last update
        new_rows = d.iloc[last_appended + 1: pos + 1]
        if len(new_rows) > 0:
            current_fit = current_fit.append(new_rows["cases"].to_numpy(),
                                              exog=new_rows[list(exog_cols)].to_numpy(),
                                              refit=False)
            last_appended = pos
        exog_future = d.iloc[pos + 1: pos + 1 + horizon_steps][list(exog_cols)]
        if len(exog_future) < horizon_steps:
            preds.append(np.nan)
            continue
        fc = current_fit.get_forecast(steps=horizon_steps, exog=exog_future.to_numpy())
        # predicted_mean may be a pandas Series or a plain ndarray depending on
        # how statsmodels wraps the result — np.asarray(...)[-1] works for both
        preds.append(np.asarray(fc.predicted_mean)[-1])

    result_labels = orig_labels[test_positions]        # map back to district_df's ORIGINAL labels
    return pd.Series(preds, index=result_labels)

# Set to None to run all 25 districts (final baseline table). Set back to a
# short list if you ever need a quick sanity-check run again.
SARIMAX_DISTRICTS = None  # None = all 25 districts

import time as _time

for horizon, steps, ycol in [("t+2", 2, "y_t2"), ("t+4", 4, "y_t4")]:
    preds_all, truth_all = [], []
    groups = feat.groupby("district")
    n_done = 0
    n_total = 25 if SARIMAX_DISTRICTS is None else len(SARIMAX_DISTRICTS)
    for dist, d in groups:
        if SARIMAX_DISTRICTS is not None and dist not in SARIMAX_DISTRICTS:
            continue
        t0 = _time.time()
        try:
            p = sarimax_forecast_district(d, steps)
            truth = d.loc[p.index, ycol]
            preds_all.append(p)
            truth_all.append(truth)
            n_done += 1
            print(f"[{horizon}] {n_done}/{n_total} done: {dist} "
                  f"({_time.time()-t0:.1f}s)", flush=True)
        except Exception as e:
            n_done += 1
            print(f"[{horizon}] {n_done}/{n_total} FAILED: {dist}: {e} "
                  f"({_time.time()-t0:.1f}s)", flush=True)
    preds_all = pd.concat(preds_all).dropna()
    truth_all = pd.concat(truth_all).loc[preds_all.index]
    r = score(truth_all, preds_all, "SARIMAX (per-district)")
    r["horizon"] = horizon
    all_results.append(r)


[t+2] 1/25 done: Ampara (6.2s)
[t+2] 2/25 done: Anuradhapura (3.7s)
[t+2] 3/25 done: Badulla (3.7s)
[t+2] 4/25 done: Batticaloa (4.4s)
[t+2] 5/25 done: Colombo (4.0s)
[t+2] 6/25 done: Galle (3.4s)
[t+2] 7/25 done: Gampaha (3.6s)
[t+2] 8/25 done: Hambantota (5.1s)
[t+2] 9/25 done: Jaffna (4.4s)
[t+2] 10/25 done: Kalutara (3.6s)
[t+2] 11/25 done: Kandy (4.6s)
[t+2] 12/25 done: Kegalle (3.6s)
[t+2] 13/25 done: Kilinochchi (3.8s)
[t+2] 14/25 done: Kurunegala (4.2s)
[t+2] 15/25 done: Mannar (4.6s)
[t+2] 16/25 done: Matale (3.5s)
[t+2] 17/25 done: Matara (3.5s)
[t+2] 18/25 done: Monaragala (7.4s)
[t+2] 19/25 done: Mullaitivu (3.9s)
[t+2] 20/25 done: Nuwara Eliya (2.4s)
[t+2] 21/25 done: Polonnaruwa (5.1s)
[t+2] 22/25 done: Puttalam (3.7s)
[t+2] 23/25 done: Ratnapura (3.1s)
[t+2] 24/25 done: Trincomalee (4.0s)
[t+2] 25/25 done: Vavuniya (4.3s)
[t+4] 1/25 done: Ampara (3.8s)
[t+4] 2/25 done: Anuradhapura (4.2s)
[t+4] 3/25 done: Badulla (4.5s)
[t+4] 4/25 done: Batticaloa (3.5s)
[t+4] 5/25 done:

In [ ]:
import pandas as pd
sarimax_rows = [r for r in all_results if r["model"] == "SARIMAX (per-district)"]
print(pd.DataFrame(sarimax_rows)[["horizon", "model", "MAE", "RMSE", "Pearson_r"]])

  horizon                   model        MAE       RMSE  Pearson_r
0     t+2  SARIMAX (per-district)  19.302402  38.655398   0.952773
1     t+4  SARIMAX (per-district)  27.983173  61.359861   0.876453
2     t+2  SARIMAX (per-district)  11.417431  21.707047   0.941653
3     t+4  SARIMAX (per-district)  15.092395  31.706181   0.872089


In [ ]:
results_df = pd.DataFrame(all_results)[["horizon", "model", "MAE", "RMSE", "Pearson_r"]]
results_df = results_df.sort_values(["horizon", "MAE"])
print(results_df.to_string(index=False))
results_df.to_csv("baseline_results_table.csv", index=False)

horizon                   model       MAE      RMSE  Pearson_r
    t+2  SARIMAX (per-district) 11.417431 21.707047   0.941653
    t+2    LSTM (global, tuned) 11.949770 24.088212   0.923538
    t+2 XGBoost (global, tuned) 12.491556 25.607441   0.917054
    t+2       Naive persistence 13.875000 30.267557   0.894990
    t+2  SARIMAX (per-district) 19.302402 38.655398   0.952773
    t+4    LSTM (global, tuned) 14.871162 29.450728   0.877458
    t+4  SARIMAX (per-district) 15.092395 31.706181   0.872089
    t+4 XGBoost (global, tuned) 15.416316 30.724379   0.871599
    t+4       Naive persistence 17.252692 39.059358   0.819615
    t+4  SARIMAX (per-district) 27.983173 61.359861   0.876453


In [ ]:
import pandas as pd

df_all = pd.DataFrame(all_results)
# keep only the LAST occurrence of each (horizon, model) pair — i.e. the most recent run
df_all = df_all.drop_duplicates(subset=["horizon", "model"], keep="last")

results_df = df_all[["horizon", "model", "MAE", "RMSE", "Pearson_r"]].sort_values(["horizon", "MAE"])
print(results_df.to_string(index=False))
results_df.to_csv("baseline_results_table.csv", index=False)

horizon                   model       MAE      RMSE  Pearson_r
    t+2  SARIMAX (per-district) 11.417431 21.707047   0.941653
    t+2    LSTM (global, tuned) 11.949770 24.088212   0.923538
    t+2 XGBoost (global, tuned) 12.491556 25.607441   0.917054
    t+2       Naive persistence 13.875000 30.267557   0.894990
    t+4    LSTM (global, tuned) 14.871162 29.450728   0.877458
    t+4  SARIMAX (per-district) 15.092395 31.706181   0.872089
    t+4 XGBoost (global, tuned) 15.416316 30.724379   0.871599
    t+4       Naive persistence 17.252692 39.059358   0.819615
